# Train SAR Ship Detector (Colab)

Runs Phase 2 (baseline training) and Phase 3 (evaluation) on a free Colab GPU.

**Before running**: Runtime -> Change runtime type -> select a GPU (T4 is fine).

## 1. Clone the repo

In [ ]:
!git clone https://github.com/ehulle117/sar-ship-detector.git
%cd sar-ship-detector

## 2. Install dependencies
Colab already ships torch/torchvision with CUDA, so just add the extras.

In [ ]:
!pip install -q pycocotools tqdm

## 3. Download and prepare the dataset

Pulls the official SSDD release from Google Drive and stages the BBox/voc_style
images + annotations into `data/JPEGImages` and `data/Annotations`, matching
what `src/dataset.py` expects.

In [ ]:
!pip install -q gdown
!gdown 1glNJUGotrbEyk43twwB9556AdngJsynZ -O Official-SSDD-OPEN.rar
!apt-get -qq install -y unrar > /dev/null
!unrar x -o+ Official-SSDD-OPEN.rar > /dev/null
!mkdir -p data/JPEGImages data/Annotations
!cp Official-SSDD-OPEN/BBox_SSDD/voc_style/JPEGImages/*.jpg data/JPEGImages/
!cp Official-SSDD-OPEN/BBox_SSDD/voc_style/Annotations/*.xml data/Annotations/
!rm -rf Official-SSDD-OPEN Official-SSDD-OPEN.rar
!ls data/JPEGImages | wc -l

## 4. Sanity check on GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 5. Train (Phase 2)
Baseline run: 10 epochs, batch size 4. Adjust as needed — should take minutes per epoch on a T4.

In [ ]:
%cd src
!python train.py --data-root ../data --epochs 10 --batch-size 4 --lr 0.005 --output-dir ../outputs

## 6. Evaluate (Phase 3)
Point at the last epoch's checkpoint.

In [ ]:
!python evaluate.py --data-root ../data --checkpoint ../outputs/checkpoint_epoch10.pt --iou-threshold 0.5 --score-threshold 0.5

## 7. Visualize sample predictions

In [ ]:
!python visualize.py --data-root ../data --checkpoint ../outputs/checkpoint_epoch10.pt --output-dir ../outputs/visualizations --num-samples 8

## 8. Download results

Checkpoints are large — usually only worth keeping the final one. Zip up the
small stuff (visualizations, metrics) to bring back locally and commit.

In [ ]:
%cd ..
!zip -r outputs_bundle.zip outputs/visualizations outputs/checkpoint_epoch10.pt
from google.colab import files
files.download('outputs_bundle.zip')